Ciao

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

In [2]:
with open('../datasets/shakespeare.txt') as f:
    shakespeare_text = f.read()

In [3]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character",standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]
encoded # this is our text encoded as integers

<tf.Tensor: shape=(1115394,), dtype=int64, numpy=array([21,  7, 10, ..., 22, 28, 12])>

In [4]:
# Each character is now mapped to an integer, starting at 2. 
# The TextVectorization layer reserved the value 0 for padding tokens, 
# and it reserved 1 for unknown characters


n_tokens = text_vec_layer.vocabulary_size() # number of distinct chars 
dataset_size = len(encoded) # total number of chars
print('number of distinct chars =', n_tokens)
print('total number of chars =', dataset_size)

vocab = text_vec_layer.get_vocabulary()
print("Vocabulary:", vocab)
int_to_word = dict(enumerate(vocab))

def decode(encoded): 
    return "".join([int_to_word[i] for i in encoded if i != 0])

number of distinct chars = 41
total number of chars = 1115394
Vocabulary: ['', '[UNK]', np.str_(' '), np.str_('e'), np.str_('t'), np.str_('o'), np.str_('a'), np.str_('i'), np.str_('h'), np.str_('s'), np.str_('r'), np.str_('n'), np.str_('\n'), np.str_('l'), np.str_('d'), np.str_('u'), np.str_('m'), np.str_('y'), np.str_('w'), np.str_(','), np.str_('c'), np.str_('f'), np.str_('g'), np.str_('b'), np.str_('p'), np.str_(':'), np.str_('k'), np.str_('v'), np.str_('.'), np.str_("'"), np.str_(';'), np.str_('?'), np.str_('!'), np.str_('-'), np.str_('j'), np.str_('q'), np.str_('x'), np.str_('z'), np.str_('3'), np.str_('&'), np.str_('$')]


In [13]:
# let's see the encoding the string 'First'

encoded_string = text_vec_layer(['First'])[0].numpy()
print(encoded_string)

# let's decode

decoded_string = decode(encoded_string) 
print(decoded_string)


[21  7 10  9  4]
first


## Dataset Preparation



### Splitting the Sequential Dataset into Multiple Windows

You cannnot just divide the string into a parts for training, validation and testig, otherwise the training set would consists of a single sequence of over a million characters, so we can’t just train the neural network directly on it: the RNN would be equivalent to a deep net with over a million layers, and we would have a single (very long) instance to train it. Instead, we will use the dataset’s window() method to convert this long sequence of characters into many smaller windows of text.

In [ ]:

encoded.numpy() # this is a numpy array of integers

# run a sliding window over the text to create sequences of 100 characters
# We set the window length to 100, but you can try tuning it: it’s easier and faster to train
# RNNs on shorter input sequences, but the RNN will not be able to learn any pattern longer
# than length, so don’t make it too small.

seq_length = 100

sequences = []

for i in range(dataset_size - seq_length):
    sequences.append(encoded[i:i+seq_length+1]) # +1 to include the next char (the one we want to predict!)
sequences = np.array(sequences) # convert to numpy array


print(sequences.shape) # (number of sequences, seq_length + 1)
print(sequences[0]) # first sequence





(1115294, 101)
[21  7 10  9  4  2 20  7  4  7 37  3 11 25 12 23  3 21  5 10  3  2 18  3
  2 24 10  5 20  3  3 14  2  6 11 17  2 21 15 10  4  8  3 10 19  2  8  3
  6 10  2 16  3  2  9 24  3  6 26 28 12 12  6 13 13 25 12  9 24  3  6 26
 19  2  9 24  3  6 26 28 12 12 21  7 10  9  4  2 20  7  4  7 37  3 11 25
 12 17  5 15  2]


In [7]:
# shuffle the sequences

np.random.shuffle(sequences)


In [14]:
print(sequences[0]) # first sequence

[13 14  2 23  3  2 24  3 10  4  7 11  3 11  4 12 23 15  4 19  2  9  5  2
  7  4  2  7  9 19  2  7  4  2  7  9  2 11  5  4 28  2 18  6  9  2  4  8
  7  9  2  4  6 26  3 11 12 23 17  2  6 11 17  2 15 11 14  3 10  9  4  6
 11 14  7 11 22  2 24  6  4  3  2 23 15  4  2  4  8  7 11  3 31 12 21  5
 10  2  4  8 17]


In [26]:
train_set = sequences[:1000000]
valid_set = sequences[1000000:1060000]
test_set = sequences[1060000:]

train_set = train_set[..., np.newaxis].astype(np.float32)
valid_set = valid_set[..., np.newaxis].astype(np.float32)
test_set = test_set[..., np.newaxis].astype(np.float32)

print("train_set shape:", train_set.shape)
print("valid_set shape:", valid_set.shape)
print("test_set shape:", test_set.shape)


train_set shape: (1000000, 101, 1)
valid_set shape: (60000, 101, 1)
test_set shape: (55294, 101, 1)


In [47]:
X_train = train_set[:,:100]
y_train = train_set[:,1:]

X_valid = valid_set[:,:100]
y_valid = valid_set[:,1:]

X_test = test_set[:,:100]
y_test = test_set[:,1:]


print(X_train.shape)
print(y_train.shape)

(1000000, 100, 1)
(1000000, 100, 1)


In [48]:
model = keras.models.Sequential([
    keras.layers.Input(shape=[None, 1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(n_tokens, activation="softmax"))
])
model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_28 (LSTM)                  │ (None, None, 20)       │         1,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_29 (LSTM)                  │ (None, None, 20)       │         3,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_13             │ (None, None, 41)       │           861 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,901 (23.05 KB)

 Trainable params: 5,901 (23.05 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, verbose=1),
]

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid), callbacks=callbacks)
model.save('saved_text_gen.keras')

Epoch 1/20
31250/31250 ━━━━━━━━━━━━━━━━━━━━ 1355s 43ms/step - loss: 2.5593 - val_loss: 2.1879
Epoch 2/20
31250/31250 ━━━━━━━━━━━━━━━━━━━━ 1428s 46ms/step - loss: 2.1603 - val_loss: 2.1064
Epoch 3/20
17318/31250 ━━━━━━━━━━━━━━━━━━━━ 10:28 45ms/step - loss: 2.1004

KeyboardInterrupt: 

In [50]:
model = keras.saving.load_model("saved_text_gen.keras")

In [56]:
encoded_string = text_vec_layer(['To be or not to b'])
print(encoded_string)

tf.Tensor([[ 4  5  2 23  3  2  5 10  2 11  5  4  2  4  5  2 23]], shape=(1, 17), dtype=int64)


In [73]:
z = text_vec_layer(['How are yo'])
prob = model.predict(z)
prob = prob[:, -1, :] # take the last time step
y_pred = tf.argmax([prob[0]], axis=-1) # take the most probable token
decode(y_pred.numpy()) # this is the predicted next character   

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


'u'

To generate new text using the char-RNN model, we could feed it some text, make the model predict the most likely next letter, add it to the end of the text, then give the extended text to the model to guess the next letter, and so on. This is called greedy decoding. But in practice this often leads to the same words being repeated over and over again. Instead, we can sample the next character randomly, with a probability  equal to the estimated probability

In [ ]:
def next_char(text, temperature=1):
    X_new = text_vec_layer([text])
    y_proba = model.predict(X_new)[0, -1:, :]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1) 
    return decode(char_id.numpy()[0]) # this is the predicted next character



def complete_text(text, n_chars=100, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

#out = complete_text("First Citizen:", temperature=0.1) 
out = complete_text("To be or not to be", temperature=0.4) 
print(out)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━